# Captioning Training (RNN/LSTM)
Notebook modular untuk preprocessing dan training decoder.

In [ ]:
import json
import sys
from pathlib import Path

import tensorflow as tf

sys.path.insert(0, str(Path('src/rnn-lstm').resolve()))
sys.path.insert(0, str(Path('src/utils').resolve()))

import preprocess_flickr8k
import train_decoder
from feature_extraction import extract_flickr8k_features, load_features
from text_utils import load_vocab, load_flickr8k_captions

In [ ]:
# Sesuaikan path ini
images_dir = Path('/path/to/Flickr8k_Dataset')
captions_file = Path('/path/to/captions.txt')
train_split = Path('/path/to/Flickr_8k.trainImages.txt')
val_split = Path('/path/to/Flickr_8k.devImages.txt')
test_split = Path('/path/to/Flickr_8k.testImages.txt')

output_dir = Path('outputs/rnn_lstm/notebook')
output_dir.mkdir(parents=True, exist_ok=True)

features_path = output_dir / 'flickr8k_features.npy'
preprocess_dir = output_dir / 'preprocess'
splits_json = preprocess_dir / 'splits.json'
vocab_path = preprocess_dir / 'vocab.json'

## Feature extraction (sekali saja)

In [ ]:
if images_dir.exists() and not features_path.exists():
    extract_flickr8k_features(
        images_dir=str(images_dir),
        output_path=str(features_path),
        encoder_name="inceptionv3",
        batch_size=32,
    )
    print("Saved features to", features_path)
elif features_path.exists():
    print("Features sudah ada:", features_path)
else:
    print("Path images_dir belum valid.")

## Preprocess captions

In [ ]:
if captions_file.exists() and train_split.exists() and val_split.exists() and test_split.exists():
    preprocess_flickr8k.preprocess(
        captions_file=captions_file,
        train_split=train_split,
        val_split=val_split,
        test_split=test_split,
        output_dir=preprocess_dir,
        min_freq=1,
    )
else:
    print("Set path captions/splits dulu.")

## Train decoder (grid kecil)

In [ ]:
if captions_file.exists() and splits_json.exists() and vocab_path.exists() and features_path.exists():
    captions = load_flickr8k_captions(str(captions_file))
    with splits_json.open("r", encoding="utf-8") as handle:
        splits = json.load(handle)
    word2idx, _ = load_vocab(vocab_path)
    features = load_features(str(features_path))

    layer_grid = [1, 2, 3]
    hidden_grid = [128, 512]

    for n_layers in layer_grid:
        for hidden_size in hidden_grid:
            config = train_decoder.DecoderExperimentConfig(
                decoder_type="lstm",
                n_layers=n_layers,
                hidden_size=hidden_size,
                embed_dim=256,
                max_seq_len=35,
                feature_dim=2048,
                learning_rate=1e-3,
                injection_method="pre",
            )
            train_decoder.train_one(
                config=config,
                captions=captions,
                splits={"train": splits["train"], "val": splits["val"], "test": splits["test"]},
                features=features,
                word2idx=word2idx,
                output_root=output_dir / "train",
                batch_size=32,
                epochs=1,
                seed=42,
            )
else:
    print("Set path preprocess dan features dulu.")